<a href="https://colab.research.google.com/github/jason-snow58/Tuning-the-Stability-of-a-Disulfide-Stabilized-Phage-VLP-by-Interface-Guided-Capsid-Engineering/blob/main/Tuning_the_Stability_of_a_Disulfide_Stabilized_Phage_VLP_by_Interface_Guided_Capsid_Engineering_Figure2_C_contact_maps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Figure 2C — residue-contact maps

Supporting analysis for *Tuning the Stability of a Disulfide-Stabilized Phage VLP by Interface-Guided Capsid Engineering*

---

### What this analysis is, in the paper

Panel C of Figure 2 draws the interaction network around I8, L23 and F116 for each of the three dimer–dimer interfaces. The point it makes is modularity: I8 feeds an N-terminal network that converges on the F116 pocket, while L23 forms a separate cluster. That is the structural reading behind targeting F116 first, then I8, then L23.

### Two things to know before reading the figures

**Node positions carry no structural meaning.** They come from a deterministic layout chosen for legibility. Distances and angles on the page are not physical quantities.

**Each map shows a subset.** A connected group of residues is drawn only where at least one of its contacts crosses between the two dimers, so self-contained pockets are omitted — a residue absent from a map is not a residue without contacts. The published panels were additionally refined by hand afterwards, which is not reproduced here.

> **A note on structure.** Unlike the Figure 1 notebook, this step is a *preserved producer*: reading, deciding, drawing and writing happen inside one operation, and it changes PyMOL state directly. It is kept in that shape deliberately, because it reproduces the retained output and restructuring its internals would put that at risk. The phase separation described in the Figure 2 notebook does not apply here, and the notebook does not pretend otherwise.

## Setup

In [ ]:
#@title Setup and input controls { display-mode: "form" }
#@markdown Run this cell first. It detects the environment, installs anything
#@markdown missing, imports everything the notebook needs, and collects the
#@markdown input settings below. Defaults reproduce the published analysis.

# ---- environment -----------------------------------------------------------
try:
    import google.colab            # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import glob
import os
import subprocess
import sys

def ensure(package, module=None):
    """Import a package, installing it first if this is Colab."""
    name = module or package
    try:
        __import__(name)
        return True
    except ImportError:
        if IN_COLAB:
            print(f"installing {package} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", package],
                           check=True)
            __import__(name)
            return True
        print(f"MISSING: {package}")
        print(f"  conda install -c conda-forge {package}")
        return False


# ---- dependencies ----------------------------------------------------------
READY = ensure("matplotlib")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
print("matplotlib", matplotlib.__version__)

# ---- input controls --------------------------------------------------------
#@markdown **Contact table.** Leave blank to search `data/` and `output/`.
CONTACT_CSV = "" #@param {type:"string"}
#@markdown **Layout seed.** Node positions carry no structural meaning; the
#@markdown same seed always produces the same arrangement.
SEED = 0 #@param {type:"integer"}
#@markdown **Formats to write.**
WRITE_SVG = True #@param {type:"boolean"}
WRITE_PNG = True #@param {type:"boolean"}
DPI = 300 #@param {type:"integer"}
OUTPUT_DIR = "output" #@param {type:"string"}

# ---- helpers ---------------------------------------------------------------
def find_input(description, patterns, search_dirs, allow_multiple=False):
    """Locate an input file, or offer an upload box in Colab.

    Patterns are tried in order, so a specific name wins over a general glob.
    Returns one path unless allow_multiple=True.
    """
    for pattern in patterns:
        hits, seen = [], set()
        for directory in search_dirs:
            # '**' searches subdirectories, which matters because each analysis
            # is committed into its own directory named after the structure.
            found = glob.glob(os.path.join(directory, pattern), recursive=True)
            for path in sorted(found):
                real = os.path.realpath(path)
                if os.path.isfile(path) and real not in seen:
                    seen.add(real)
                    hits.append(path)
        if hits:
            print(f"input: matched {pattern!r}")
            for h in hits:
                print("   ", h)
            if len(hits) > 1 and not allow_multiple:
                print("   using the first; set the variable directly to choose another")
                return hits[0]
            return hits if allow_multiple else hits[0]

    if IN_COLAB:
        from google.colab import files
        print(f"Upload {description}:")
        uploaded = files.upload()
        if not uploaded:
            return None
        names = list(uploaded)
        return names if allow_multiple else names[0]

    print(f"Nothing found for {description}.")
    print("Searched:", ", ".join(search_dirs))
    print("Patterns:", ", ".join(repr(p) for p in patterns))
    return None


def require(value, what):
    if value is None:
        raise SystemExit(f"No input for {what}. See the message above.")
    return value


def deliver(paths):
    """Report finished files, and download them in Colab."""
    paths = [paths] if isinstance(paths, str) else list(paths)
    for p in paths:
        size = os.path.getsize(p) if os.path.exists(p) else 0
        print(f"   {p}  ({size:,} bytes)")
    if IN_COLAB:
        from google.colab import files
        for p in paths:
            files.download(p)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print()
print("Colab" if IN_COLAB else "local Jupyter", "| python", sys.version.split()[0])
print("output directory:", os.path.abspath(OUTPUT_DIR))

## The analysis code

These cells write the analysis package into the working directory, so the notebook is self-contained and nothing has to be fetched. This is the same source that accompanies the manuscript — read it if you want to check the calculation, or run straight past it.

In [ ]:
os.makedirs("capsid", exist_ok=True)
print("package directory ready")

In [ ]:
%%writefile capsid/__init__.py
"""Interface and contact analysis for ssRNA phage capsids.

WHICH PIPELINES ARE SEPARATED, AND WHICH ARE NOT

Only the contact-classification path implements the phase separation in its
call graph:

    decide.analyse(pdb)                  reads the structure; classifies; once
    render.render_analysis(decision)     pure; produces every finished file
    validate.validate_rendering(...)     three derivations compared row by row
    effect.commit_artifacts(...)         transport only; cannot render

``effect`` does not import ``render`` and never receives an analysis record, so
no output path can decide or render once writing has begun. Files are staged
and committed as a set, so a failed run leaves the previous output untouched
rather than a directory holding half of one run and half of another.

The three PyMOL and figure modules -- ``pymol_contacts``, ``calpha_svg`` and
``network_map`` -- are a different shape. Each interleaves reading, deciding,
drawing and writing in a single operation, and each writes files or changes
PyMOL state directly. They are kept that way deliberately, because they
reproduce the retained outputs and restructuring their internals would put that
at risk. The guarantees above describe the classification path only.
"""

import importlib

__all__ = ['artifacts', 'calpha_svg', 'decide', 'effect', 'manifest',
           'network_map', 'palette', 'pymol_contacts', 'records', 'render',
           'validate']


def __getattr__(name):
    """Import submodules on first use.

    Importing the package must not require every optional dependency. A
    notebook that only classifies contacts needs neither matplotlib nor PyMOL,
    and should not be made to install them to say ``import capsid``.
    """
    if name in __all__:
        module = importlib.import_module(f'.{name}', __name__)
        globals()[name] = module
        return module
    raise AttributeError(f'module {__name__!r} has no attribute {name!r}')


def __dir__():
    return sorted(__all__)

In [ ]:
%%writefile capsid/network_map.py
"""Residue-contact map figures for the AP205 hydrophobic-patch residues.

Reads a contact table produced by pymol_contacts.py and draws one network per
dimer-dimer interface: residues as discs sized by contact count, coloured by
subunit, with the contacting atoms labelled around each disc and dashed lines
connecting contacting atom pairs.

TWO THINGS A READER OF THESE FIGURES NEEDS TO KNOW.

Node positions carry no structural meaning. They come from a deterministic
golden-angle layout chosen only for legibility. The same ``seed`` always gives
the same picture and a different seed gives a different arrangement of the same
network; distances and angles on the page are not physical quantities.

What is drawn is a subset. A connected group of residues is shown only when at
least one of its contacts crosses between the two dimers, so self-contained
pockets that do not contribute to the interface are omitted. A residue absent
from a map is not necessarily a residue without contacts.
"""

import csv
import math
from collections import defaultdict

__version__ = '1.2'

# The three hydrophobic-patch residues the figure is about.
RESIDUES_OF_INTEREST = {'PHE': (116,), 'ILE': (8,), 'LEU': (23,)}

# AP205 trimer-of-dimers: six chains, three dimers. Colour follows the dimer,
# so the same colour always means the same structural role.
DIMERS = {'H/LE': ('H', 'LE'), 'ME/YC': ('ME', 'YC'), 'BB/NE': ('BB', 'NE')}

CHAIN_COLORS = {'H': 'cyan', 'ME': 'cyan',
                'LE': 'green', 'YC': 'green',
                'BB': 'purple', 'NE': 'purple'}

INTER_DIMER_STYLE = {'color': 'darkred', 'linewidth': 3.0, 'alpha': 0.7}
INTRA_DIMER_STYLE = {'color': 'gray', 'linewidth': 1.0, 'alpha': 0.3}

REQUIRED_COLUMNS = ('contact_group', 'distance_angstrom', 'source_segi',
                    'source_resi', 'source_resn', 'source_atom',
                    'target_segi', 'target_resi', 'target_resn', 'target_atom')


# --------------------------------------------------------------------------
# DECIDE -- read the contact CSV once, classify every contact
# --------------------------------------------------------------------------

class Contact:
    __slots__ = ('source_res', 'target_res', 'source_atom', 'target_atom',
                 'source_chain', 'target_chain', 'is_inter_dimer', 'interface',
                 'source_is_roi', 'target_is_roi', 'distance')

    def __init__(self, **kw):
        for k, v in kw.items():
            setattr(self, k, v)


def dimer_of(chain):
    for name, chains in DIMERS.items():
        if chain in chains:
            return name
    return None


def interface_of(source_chain, target_chain):
    """Which dimer-dimer interface a contact sits on, or None if intra-dimer."""
    a, b = dimer_of(source_chain), dimer_of(target_chain)
    if a is None or b is None or a == b:
        return None
    lo, hi = sorted((a, b))
    return f'{lo} <-> {hi}'


def residue_label(chain, resi, resn):
    return f'{chain}-{resn}{resi}'


def is_roi(resn, resi):
    return resi in RESIDUES_OF_INTEREST.get(resn, ())


def read_contacts(csv_path):
    """Parse the contact CSV into Contact records. The only read of the world.

    Residue numbers are coerced to int explicitly, so that the
    residue-of-interest test cannot depend on how the file happened to be
    written.
    """
    with open(csv_path, newline='', encoding='utf-8-sig') as fh:
        reader = csv.DictReader(fh)
        missing = [c for c in REQUIRED_COLUMNS if c not in (reader.fieldnames or [])]
        if missing:
            raise ValueError(f'{csv_path}: missing column(s) {missing}; '
                             f'found {reader.fieldnames}')
        records = []
        for lineno, row in enumerate(reader, start=2):
            try:
                s_resi = int(row['source_resi'])
                t_resi = int(row['target_resi'])
            except (TypeError, ValueError):
                raise ValueError(f'{csv_path} line {lineno}: non-integer residue '
                                 f'number {row["source_resi"]!r}/{row["target_resi"]!r}')
            s_chain, t_chain = row['source_segi'], row['target_segi']
            records.append(Contact(
                source_res=residue_label(s_chain, s_resi, row['source_resn']),
                target_res=residue_label(t_chain, t_resi, row['target_resn']),
                source_atom=row['source_atom'], target_atom=row['target_atom'],
                source_chain=s_chain, target_chain=t_chain,
                is_inter_dimer=dimer_of(s_chain) != dimer_of(t_chain),
                interface=interface_of(s_chain, t_chain),
                source_is_roi=is_roi(row['source_resn'], s_resi),
                target_is_roi=is_roi(row['target_resn'], t_resi),
                distance=row['distance_angstrom']))
    return records


def contacts_touching_roi(contacts):
    return [c for c in contacts if c.source_is_roi or c.target_is_roi]


def contacts_on_interface(contacts, interface):
    """Every contact whose two chains both belong to this interface's dimers."""
    chains = set()
    for dimer in interface.split(' <-> '):
        chains.update(DIMERS[dimer])
    return [c for c in contacts
            if c.source_chain in chains and c.target_chain in chains]


class Network:
    """One interface's contact network."""

    def __init__(self, interface, contacts):
        self.interface = interface
        self.edges = contacts
        self.contact_count = defaultdict(int)
        self.atoms = defaultdict(set)
        self.chain = {}
        self.is_roi = {}
        for c in contacts:
            for res, atom, roi, chain in (
                    (c.source_res, c.source_atom, c.source_is_roi, c.source_chain),
                    (c.target_res, c.target_atom, c.target_is_roi, c.target_chain)):
                self.contact_count[res] += 1
                self.atoms[res].add(atom)
                self.chain[res] = chain
                self.is_roi[res] = roi

    @property
    def residues(self):
        return list(self.contact_count)


def keep_interface_clusters(network):
    """Keep a connected cluster only if it reaches across the interface.

    Inclusion is decided per CLUSTER, not per contact. Residues and their
    contacts form an undirected graph; a cluster survives if any one of its
    contacts crosses between the two dimers. That keeps the intradimer residues
    packing around a genuine bridge, while dropping self-contained pockets that
    never reach the partner dimer -- in AP205, the H-PHE116 pocket and the
    isolated NE-ILE8 and NE-LEU23 pockets.

    Deciding per contact instead would discard the surrounding residues as well,
    leaving the bridging contacts without the environment that explains them.
    """
    adjacency = defaultdict(set)
    on_bridge = set()
    for e in network.edges:
        adjacency[e.source_res].add(e.target_res)
        adjacency[e.target_res].add(e.source_res)
        if e.is_inter_dimer:
            on_bridge.update((e.source_res, e.target_res))

    visited, kept = set(), set()
    for start in adjacency:
        if start in visited:
            continue
        cluster, stack = set(), [start]
        while stack:
            node = stack.pop()
            if node in visited:
                continue
            visited.add(node)
            cluster.add(node)
            stack.extend(adjacency[node] - visited)
        if cluster & on_bridge:
            kept |= cluster

    surviving = [e for e in network.edges
                 if e.source_res in kept and e.target_res in kept]
    return Network(network.interface, surviving)


def build_networks(contacts):
    """interface name -> filtered Network, for every interface present."""
    roi = contacts_touching_roi(contacts)
    interfaces = sorted({c.interface for c in roi if c.interface})
    out = {}
    for name in interfaces:
        full = Network(name, contacts_on_interface(roi, name))
        filtered = keep_interface_clusters(full)
        if filtered.contact_count:
            out[name] = filtered
    return out


# --------------------------------------------------------------------------
# RENDER -- layout and drawing, both pure functions of the Network
# --------------------------------------------------------------------------

GOLDEN_ANGLE = math.pi * (3.0 - math.sqrt(5.0))


def layout(network, seed=0, roi_radius=2.0, satellite_radius=1.2,
           orphan_radius=4.0):
    """Deterministic node placement.

    Hotspot residues are spread evenly around a circle, grouped by chain.
    Every other residue is placed on a golden-angle spiral around the first
    hotspot it contacts, which spreads satellites without them colliding and
    gives the same picture on every run.

    These positions are for legibility only and carry no structural meaning.
    """
    pos = {}
    residues = sorted(network.residues)
    roi = [r for r in residues if network.is_roi[r]]
    rest = [r for r in residues if not network.is_roi[r]]

    by_chain = defaultdict(list)
    for r in roi:
        by_chain[network.chain[r]].append(r)

    chains = sorted(by_chain)
    step = 2 * math.pi / len(chains) if chains else 0.0
    for i, chain in enumerate(chains):
        members = sorted(by_chain[chain])
        base = i * step
        for j, res in enumerate(members):
            angle = base + (j - len(members) / 2) * 0.4 if len(members) > 1 else base
            pos[res] = (roi_radius * math.cos(angle), roi_radius * math.sin(angle))

    neighbours = defaultdict(list)
    for e in network.edges:
        for a, b in ((e.source_res, e.target_res), (e.target_res, e.source_res)):
            if not network.is_roi[a] and network.is_roi[b]:
                neighbours[a].append(b)

    counter = defaultdict(int)
    for k, res in enumerate(rest):
        anchors = sorted(neighbours.get(res, ()))
        if anchors and anchors[0] in pos:
            anchor = anchors[0]
            n = counter[anchor]
            counter[anchor] += 1
            angle = (seed + n) * GOLDEN_ANGLE
            radius = satellite_radius * (1.0 + 0.18 * n)
            ax, ay = pos[anchor]
            pos[res] = (ax + radius * math.cos(angle), ay + radius * math.sin(angle))
        else:
            angle = (seed + k) * GOLDEN_ANGLE
            pos[res] = (orphan_radius * math.cos(angle),
                        orphan_radius * math.sin(angle))
    return pos


def draw(network, seed=0, figsize=(14, 14)):
    """Draw one interface network. Returns a matplotlib Figure."""
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from matplotlib.patches import Circle

    pos = layout(network, seed=seed)
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_aspect('equal')
    ax.axis('off')

    xs = [p[0] for p in pos.values()]
    ys = [p[1] for p in pos.values()]
    margin = 1.5
    ax.set_xlim(min(xs) - margin, max(xs) + margin)
    ax.set_ylim(min(ys) - margin, max(ys) + margin)

    # residue-level connections, under the discs
    for e in network.edges:
        if e.source_res in pos and e.target_res in pos:
            style = INTER_DIMER_STYLE if e.is_inter_dimer else INTRA_DIMER_STYLE
            x1, y1 = pos[e.source_res]
            x2, y2 = pos[e.target_res]
            ax.plot([x1, x2], [y1, y2], zorder=1, **style)

    peak = max(network.contact_count.values())
    atom_pos = {}
    for res, (x, y) in pos.items():
        roi = network.is_roi[res]
        lo, hi, edge_w = (0.30, 0.40, 3) if roi else (0.18, 0.25, 2)
        radius = lo + (hi - lo) * (network.contact_count[res] / peak)
        color = CHAIN_COLORS[network.chain[res]]

        if roi:
            ax.add_patch(Circle((x, y), radius * 1.15, facecolor=color,
                                edgecolor='none', alpha=0.3, zorder=9))
        ax.add_patch(Circle((x, y), radius, facecolor=color, edgecolor='black',
                            linewidth=edge_w, alpha=0.8, zorder=10))
        ax.text(x, y, res, ha='center', va='center', zorder=15,
                fontsize=10 if roi else 7,
                fontweight='bold' if roi else 'normal')

        atoms = sorted(network.atoms[res])
        for i, atom in enumerate(atoms):
            angle = 2 * math.pi * i / len(atoms)
            ax_, ay_ = x + radius * 0.75 * math.cos(angle), y + radius * 0.75 * math.sin(angle)
            atom_pos[(res, atom)] = (ax_, ay_)
            ax.text(ax_, ay_, atom, ha='center', va='center', zorder=12,
                    fontsize=7 if roi else 5,
                    bbox=dict(boxstyle='round,pad=0.15', facecolor='white',
                              edgecolor='gray', alpha=0.9, linewidth=0.5))

    # atom-level connections, over the discs
    for e in network.edges:
        a, b = (e.source_res, e.source_atom), (e.target_res, e.target_atom)
        if a in atom_pos and b in atom_pos:
            x1, y1 = atom_pos[a]
            x2, y2 = atom_pos[b]
            if e.is_inter_dimer:
                ax.plot([x1, x2], [y1, y2], color='red', linewidth=1.5,
                        alpha=0.6, zorder=5)
            else:
                ax.plot([x1, x2], [y1, y2], color='gray', linewidth=0.5,
                        alpha=0.3, zorder=5)

    roi_names = ', '.join(f'{resn}{resi}'
                          for resn, resis in RESIDUES_OF_INTEREST.items()
                          for resi in resis)
    ax.set_title(f'Interface: {network.interface}\nFocusing on: {roi_names}\n'
                 f'(interface residues only)',
                 fontsize=14, fontweight='bold', pad=20)

    handles = [mpatches.Patch(facecolor=CHAIN_COLORS[c], edgecolor='black',
                              label=f'Chain {c}')
               for c in sorted(set(network.chain.values()))]
    handles += [
        plt.Line2D([0], [0], color='white', marker='o', markersize=10,
                   markerfacecolor='gray', markeredgecolor='black',
                   markeredgewidth=3, label='Hotspot residue', linestyle=''),
        plt.Line2D([0], [0], color='darkred', linewidth=3, label='Inter-dimer contact'),
        plt.Line2D([0], [0], color='gray', linewidth=1, alpha=0.5,
                   label='Intra-dimer contact'),
    ]
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(0, 1),
              fontsize=9, framealpha=0.9)
    fig.tight_layout()
    return fig


# --------------------------------------------------------------------------
# EFFECT
# --------------------------------------------------------------------------

def safe_name(interface):
    return interface.replace('/', '_').replace(' <-> ', '_to_').replace(' ', '_')


def write_figures(networks, output_dir, seed=0, formats=('svg', 'png'), dpi=300):
    """Draw and save every interface. Returns the paths written, in order."""
    import os
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    os.makedirs(output_dir, exist_ok=True)
    tag = __version__.replace('.', '_')
    written = []
    for interface in sorted(networks):
        fig = draw(networks[interface], seed=seed)
        stem = f'interface_{safe_name(interface)}_v{tag}'
        for fmt in formats:
            path = os.path.join(output_dir, f'{stem}.{fmt}')
            fig.savefig(path, format=fmt, bbox_inches='tight',
                        **({'dpi': dpi} if fmt == 'png' else {}))
            written.append(path)
        plt.close(fig)
    return written

In [ ]:
if "." not in sys.path:
    sys.path.insert(0, ".")
import capsid
print("analysis package ready:", ", ".join(capsid.__all__))

## Draw the maps

One network per dimer–dimer interface.

In [ ]:
from capsid import network_map as nm
from IPython.display import display

csv_path = globals().get("_found") or CONTACT_CSV or find_input(
    "the contact table",
    ["contacts.csv", "**/contacts.csv", "*contact*.csv", "*.csv"],
    ["data", "output", "."])
require(csv_path, "the contact table")

contacts = nm.read_contacts(csv_path)
touching = nm.contacts_touching_roi(contacts)
networks = nm.build_networks(contacts)

print(f"contacts read              : {len(contacts)}")
print(f"touching I8 / L23 / F116   : {len(touching)}")
print(f"interfaces with a network  : {len(networks)}")
for name in sorted(networks):
    n = networks[name]
    print(f"   {name:<24} residues={len(n.residues):3d} edges={len(n.edges):3d}")

formats = tuple(f for f, on in (("svg", WRITE_SVG), ("png", WRITE_PNG)) if on)
written = nm.write_figures(networks, OUTPUT_DIR, seed=SEED,
                           formats=formats, dpi=DPI)

for name in sorted(networks):
    figure = nm.draw(networks[name], seed=SEED, figsize=(11, 11))
    display(figure)
    plt.close(figure)

deliver(written)